[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/graph_theory/02_traversal_and_connectivity/first_principles.ipynb)

# Topic 02: Traversal and Connectivity

## 1. First-Principles Intuition & Motivation

A graph stores *local* information — each vertex knows only its neighbors. Yet the questions we care about are *global*: Can I get from $u$ to $v$? How many hops does it take? Is the network one piece or several? Does the dependency structure contain a deadlock (cycle)?

Traversal algorithms bridge this gap. Starting at a source, they repeatedly apply the only move a graph allows — step across an edge — while bookkeeping which vertices have been seen. Two natural disciplines emerge:

- **Breadth-first search (BFS)**: explore like a ripple in a pond, visiting all vertices at distance 1, then 2, then 3, …
- **Depth-first search (DFS)**: explore like a maze runner with a ball of string, pushing forward until stuck, then backtracking.

Both visit exactly the reachable set and both run in linear time $O(n+m)$; the difference lies entirely in *the order of visits* — and that order is what encodes distances (BFS) or hierarchy (DFS).

### Why the order matters

- BFS order sorts vertices by distance from the source; the traversal *is* a shortest-path computation for unweighted graphs.
- DFS order nests explorations inside each other like parentheses; the nesting detects cycles, linearizes dependency graphs, and decomposes digraphs into strongly connected components.

The queue-versus-stack distinction — a one-word change in the pseudocode — is thus one of the highest leverage design decisions in algorithmics.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition (Walk, path, cycle).** A **walk** of length $k$ is a sequence $v_0, v_1, \dots, v_k$ with $\{v_{i-1}, v_i\} \in E$ for all $i$. A **path** is a walk with no repeated vertices. A **cycle** is a walk with $v_0 = v_k$, $k \ge 3$, and all other vertices distinct.

**Definition (Distance).** $d(u,v)$ is the minimum length of a $u$–$v$ path, with $d(u,v) = \infty$ if none exists.

**Definition (Connected).** An undirected graph is **connected** if $d(u,v) \lt \infty$ for every pair $u, v$. A **connected component** is a maximal connected subgraph.

**Definition (Strong connectivity).** A digraph is **strongly connected** if for every ordered pair $(u,v)$ there is a directed path from $u$ to $v$. Its **strongly connected components (SCCs)** are the maximal strongly connected subgraphs.

**Definition (Eccentricity, diameter, radius).** $\operatorname{ecc}(v) = \max_u d(v,u)$; $\operatorname{diam}(G) = \max_v \operatorname{ecc}(v)$; $\operatorname{rad}(G) = \min_v \operatorname{ecc}(v)$.

**Definition (DAG and topological order).** A **directed acyclic graph (DAG)** is a digraph with no directed cycle. A **topological order** is a linear ordering $v_1 \prec v_2 \prec \dots \prec v_n$ such that every edge $(v_i, v_j)$ satisfies $i \lt j$.

**Definition (Eulerian circuit / trail).** An **Eulerian circuit** is a closed walk using every edge exactly once; an **Eulerian trail** is an open walk doing the same.

**Theorem statements proved below.**

1. *(BFS correctness)* BFS from $s$ computes $\mathrm{dist}[v] = d(s,v)$ for every vertex $v$.
2. *(Component structure)* Reachability is an equivalence relation; components partition $V$.
3. *(Metric)* $d(\cdot,\cdot)$ is a metric on the vertex set of a connected graph.
4. *(DAG linearization)* A digraph has a topological order iff it is acyclic.
5. *(Euler, 1736)* A connected graph has an Eulerian circuit iff every vertex has even degree.

### The two traversal algorithms

```
BFS(G, s):                              DFS(G):
    dist[v] ← ∞ for all v                   color[v] ← WHITE for all v; time ← 0
    dist[s] ← 0; Q ← queue containing s     for each v ∈ V:
    while Q not empty:                          if color[v] = WHITE: DFS-Visit(v)
        u ← Q.dequeue()
        for v ∈ Adj(u):                     DFS-Visit(u):
            if dist[v] = ∞:                     time ← time+1; d[u] ← time; color[u] ← GRAY
                dist[v] ← dist[u] + 1           for v ∈ Adj(u):
                parent[v] ← u                       if color[v] = WHITE:
                Q.enqueue(v)                            parent[v] ← u; DFS-Visit(v)
                                                time ← time+1; f[u] ← time; color[u] ← BLACK
```

Both algorithms touch each vertex once and scan each adjacency list once: time $O(n+m)$, space $O(n)$ beyond the graph.

**Theorem (Parenthesis theorem).** In any DFS, for any two vertices $u, v$ the intervals $[d(u), f(u)]$ and $[d(v), f(v)]$ are either disjoint or one contains the other; containment holds iff one vertex is a descendant of the other in the DFS forest.

**Theorem (White-path theorem).** Vertex $v$ becomes a descendant of $u$ in the DFS forest iff at time $d(u)$ there is a path from $u$ to $v$ consisting entirely of white (undiscovered) vertices.

**Theorem (Cycle detection).** A digraph is acyclic iff DFS produces no back edge (an edge $(u,v)$ with $v$ gray, i.e., an ancestor of $u$).

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1: BFS computes exact shortest distances

**Theorem.** After running BFS from $s$, $\mathrm{dist}[v] = d(s,v)$ for every vertex $v$.

**Proof.**

*Step 1 — upper bound* $\mathrm{dist}[v] \ge d(s,v)$: whenever BFS sets $\mathrm{dist}[v] = \mathrm{dist}[u] + 1$ via edge $\{u,v\}$, an $s$–$v$ walk of that length exists (extend the walk to $u$ by one edge, by induction). A walk of length $\ell$ implies $d(s,v) \le \ell$.

*Step 2 — queue monotonicity*: the values $\mathrm{dist}[\cdot]$ of vertices in the queue are nondecreasing from front to back, and at any moment differ by at most 1. This holds initially ($\{s\}$) and is preserved: dequeuing $u$ enqueues only vertices with value $\mathrm{dist}[u]+1$ at the back. (Induction on dequeue operations.)

*Step 3 — lower bound* $\mathrm{dist}[v] \le d(s,v)$, by induction on $d(s,v)$: true for $d = 0$. Suppose it holds for all vertices at distance $k$, and let $d(s,v) = k+1$ with penultimate vertex $u$ on a shortest path, $d(s,u) = k$. By induction $\mathrm{dist}[u] = k$ (combining both bounds). When $u$ is dequeued, edge $\{u,v\}$ is scanned; either $v$ is still unvisited and receives $\mathrm{dist}[v] = k+1$, or $v$ was already assigned some value earlier. By monotonicity every value assigned before that moment is at most $k+1$. Either way $\mathrm{dist}[v] \le k+1$. $\blacksquare$

$$
\boxed{\mathrm{dist}_{\mathrm{BFS}}[v] = d(s,v) \quad \text{for unweighted graphs}}
$$

### Proof 2: Components partition the vertex set

**Theorem.** Define $u \sim v$ iff there is a $u$–$v$ path (with $v \sim v$ always). Then $\sim$ is an equivalence relation, and its classes are exactly the connected components; in particular the components partition $V$.

**Proof.**

- *Reflexive*: the length-0 walk gives $v \sim v$.
- *Symmetric*: reversing an undirected $u$–$v$ path gives a $v$–$u$ path.
- *Transitive*: concatenating a $u$–$v$ walk and a $v$–$w$ walk gives a $u$–$w$ walk; any $u$–$w$ walk contains a $u$–$w$ path (delete the cycle between repeated visits of any repeated vertex, and induct on length).

Equivalence classes are disjoint and cover $V$, so they partition $V$. Each class is connected (its members are pairwise joined by paths) and maximal: any vertex joined by an edge to a class member is in the same class. Hence classes = components. $\blacksquare$

**Algorithmic corollary.** Repeatedly running BFS/DFS from an unvisited vertex labels all components in total time $O(n+m)$, since each edge is scanned only within its own component.

### Proof 3: Graph distance is a metric

**Theorem.** On a connected graph, $d$ satisfies (i) $d(u,v) \ge 0$ with equality iff $u = v$; (ii) $d(u,v) = d(v,u)$; (iii) $d(u,w) \le d(u,v) + d(v,w)$.

**Proof.** (i) Lengths are nonnegative integers; only the empty path has length 0, and it requires $u = v$. (ii) Undirected paths reverse. (iii) Concatenating a shortest $u$–$v$ path and a shortest $v$–$w$ path yields a $u$–$w$ *walk* of length $d(u,v) + d(v,w)$; extracting a path from the walk (Proof 2) can only shorten it, and $d(u,w)$ is a minimum over all paths, so $d(u,w) \le d(u,v) + d(v,w)$. $\blacksquare$

The triangle inequality is the engine behind distance-based lower bounds in search (A*, Topic 04) and behind diameter approximation via double-BFS sweeps.

### Proof 4: Topological order exists iff the digraph is acyclic

**Theorem.** A digraph $G$ admits a topological order $\iff$ $G$ has no directed cycle.

**Proof.**

($\Rightarrow$) If $v_{i_1} \to v_{i_2} \to \dots \to v_{i_k} \to v_{i_1}$ were a cycle, a topological order would force $i_1 \lt i_2 \lt \dots \lt i_k \lt i_1$ — impossible.

($\Leftarrow$) First, *every DAG has a source* (in-degree 0): walk backwards along in-edges; if no source existed, the walk would run forever, but after $n$ steps some vertex repeats, exhibiting a cycle — contradiction.

Now induct on $n$. A single vertex is trivially ordered. For $n \gt 1$, pick a source $u$, place it first, and delete it. Deleting a vertex cannot create a cycle, so $G - u$ is a DAG on $n-1$ vertices with a topological order by induction. Prepending $u$ is valid because every edge incident to $u$ leaves $u$ (it had in-degree 0). $\blacksquare$

**Kahn's algorithm** is this proof made executable: repeatedly remove a current source and decrement its out-neighbors' in-degrees, in $O(n+m)$ time. Equivalently, sorting by *decreasing DFS finishing time* yields a topological order, because in a DAG every edge $(u,v)$ satisfies $f(u) \gt f(v)$.

### Proof 5: Euler's theorem

**Theorem (Euler, 1736).** A connected graph (or multigraph) $G$ has an Eulerian circuit $\iff$ every vertex has even degree.

**Proof.**

($\Rightarrow$) Fix an Eulerian circuit. Each time the circuit passes through a vertex $v$, it consumes exactly one edge entering and one edge leaving $v$. All of $\deg(v)$ is consumed by such visits (the start vertex pairs its initial departure with its final arrival), so $\deg(v)$ is a sum of 2's — even.

($\Leftarrow$) Assume all degrees even; induct on the number of edges $m$. The claim is trivial for $m = 0$.

1. *A closed walk exists*: start anywhere and walk along unused edges. At every vertex other than the start, an even degree means arriving consumes an odd count so far, leaving an unused edge to depart on. The walk can only get stuck at the start vertex, having formed a closed walk $C$ using distinct edges.
2. *Remove and recurse*: delete the edges of $C$. Every vertex loses an even number of incident edges, so degrees stay even; each nonempty component $H_i$ of the remainder has fewer edges and (by induction) an Eulerian circuit $C_i$.
3. *Splice*: since $G$ is connected, each $H_i$ shares at least one vertex with $C$. Traverse $C$, and upon first reaching a shared vertex of an unspliced $H_i$, detour around $C_i$ before continuing. The result is a single closed walk using every edge of $G$ exactly once. $\blacksquare$

$$
\boxed{\text{Eulerian circuit} \iff \text{connected and all degrees even}}
$$

**Corollary (Eulerian trail).** A connected graph has an open Eulerian trail iff exactly two vertices have odd degree (the trail must start at one and end at the other — add a phantom edge between them and apply the theorem). Königsberg has four odd vertices: no trail, no circuit.

### Proof 6: Cycle detection via back edges

**Theorem.** A digraph has a directed cycle $\iff$ every DFS of it produces a back edge.

**Proof.**

($\Leftarrow$) A back edge $(u,v)$ points to a gray ancestor $v$ of $u$; the tree path $v \rightsquigarrow u$ plus $(u,v)$ closes a directed cycle.

($\Rightarrow$) Let $C$ be a cycle and $v$ the first vertex of $C$ discovered by DFS. At time $d(v)$, every other vertex of $C$ is white, and $C$ provides a white path from $v$ to its cycle-predecessor $u$. By the white-path theorem, $u$ becomes a descendant of $v$, so when the edge $(u,v)$ is scanned, $v$ is still gray (its interval contains $u$'s by the parenthesis theorem). Hence $(u,v)$ is classified as a back edge. $\blacksquare$

This is how build systems, package managers, and spreadsheet engines detect circular dependencies in linear time.

## 4. Computational & Algorithmic Insights

### Complexity summary

| Task | Algorithm | Time | Certificate produced |
|---|---|---|---|
| Reachable set from $s$ | BFS or DFS | $O(n+m)$ | traversal tree |
| Unweighted shortest paths | BFS | $O(n+m)$ | distance layers + parent tree |
| Connected components | repeated BFS/DFS | $O(n+m)$ | component labels |
| Bipartiteness / odd cycle | BFS 2-coloring | $O(n+m)$ | 2-coloring or odd cycle |
| Cycle detection (digraph) | DFS | $O(n+m)$ | back edge or DAG proof |
| Topological sort | DFS finish times / Kahn | $O(n+m)$ | linear order |
| Strongly connected components | Tarjan / Kosaraju | $O(n+m)$ | SCC labels + condensation |
| Eulerian circuit | Hierholzer | $O(n+m)$ | explicit circuit |

Every row is *linear time* — a striking fact: the deepest order-theoretic structure of a digraph (its SCC condensation) costs no more than reading the input.

### Practical notes

- **Queue vs. stack**: replacing BFS's FIFO queue by a LIFO stack yields (iterative) DFS — the entire behavioral difference of the two algorithms is the container discipline.
- **Bipartiteness test**: BFS-color vertices by layer parity; an edge inside a layer closes an odd cycle, certifying non-bipartiteness (used in Topic 05).
- **Connectivity via linear algebra**: $G$ is connected iff $(I + A)^{n-1}$ has no zero entry — elegant but $O(n^{\omega} \log n)$; traversal does it in $O(n+m)$. The spectral test $\lambda_2(L) \gt 0$ (Topic 06) is a third route.
- **Memory-constrained BFS**: frontier-based implementations store only two layers at a time; web-scale crawlers and GPU BFS (direction-optimizing BFS) exploit this.

### BFS 2-coloring as a bipartiteness test

Color each vertex by the parity of its BFS layer: $c(v) = d(s,v) \bmod 2$. Every edge joins vertices whose distances differ by at most 1 (layer structure), so an edge is either *cross-layer* (endpoints of different parity — consistent with bipartiteness) or *intra-layer* (same parity).

- If no intra-layer edge exists, the parity classes form a valid 2-coloring: the graph is bipartite.
- If an intra-layer edge $\{u, v\}$ exists, then the BFS-tree paths from $s$ to $u$ and to $v$, joined by $\{u,v\}$, contain a cycle of length $d(s,u) + d(s,v) + 1$, which is **odd** — a certificate of non-bipartiteness.

Either way, one BFS pass returns a constructive certificate in $O(n+m)$ time. Topic 05 uses this to prove König's characterization: bipartite $\iff$ no odd cycle.

### Verification strategy

When implementing or debugging traversal code, check these invariants:

- **BFS layer condition**: for every edge $\{u,v\}$, assert $\lvert \mathrm{dist}[u] - \mathrm{dist}[v] \rvert \le 1$; any violation reveals a broken queue discipline.
- **Parent consistency**: $\mathrm{dist}[v] = \mathrm{dist}[\mathrm{parent}[v]] + 1$ for every non-source vertex reached.
- **DFS parenthesization**: sorted intervals $[d(u), f(u)]$ must nest or be disjoint; partial overlap indicates timestamp bugs.
- **Handshaking audit**: a full traversal over adjacency lists should perform exactly $2m$ (undirected) or $m$ (directed) edge scans.
- **Component sanity**: the sum of component sizes must equal $n$; recomputing components from the complement of a small graph cross-checks the labeling.
- **Topological order check**: verify every edge $(u,v)$ has $\mathrm{pos}[u] \lt \mathrm{pos}[v]$ — an $O(m)$ certificate validation.

## 5. Real-World Physics & AI/ML Applications

### Physics, epidemics, and infrastructure

- **Epidemic spreading / contact tracing**: an infection seeded at $s$ reaches exactly the BFS layers of the contact graph; layer index = generation of infection. Ring vaccination targets $L_1 \cup L_2$.
- **Percolation**: in statistical physics, connectivity of random subgraphs (bond percolation) undergoes a phase transition; component-finding is the computational core of percolation simulations.
- **Power grids and cascading failures**: after removing failed lines, connected-component analysis determines islanding; strong connectivity governs controllability of directed flow networks.
- **Circuit verification**: signal reachability in a netlist is directed reachability; combinational loops are directed cycles caught by DFS.

### AI and machine learning

- **GNN receptive fields**: after $k$ rounds of message passing, the embedding of $v$ depends exactly on the BFS ball $B_k(v) = \{u : d(u,v) \le k\}$. Small-world graphs have exploding balls ($\lvert B_k \rvert \approx \bar{d}^k$), causing *over-squashing* — exponentially much information forced through fixed-size vectors.
- **Web crawling & knowledge graph population**: crawlers are BFS frontiers over the directed web graph; politeness constraints and prioritization turn it into best-first search.
- **Deep learning schedulers**: autograd engines (PyTorch, JAX) topologically sort the computation DAG to order forward execution and reverse it for backpropagation; cycle detection guards against invalid graph construction.
- **Curriculum and dependency ordering**: prerequisite structures (like this repository's modules) are DAGs; any valid study order is a topological sort.
- **Six degrees of separation**: with $n \approx 8 \times 10^9$ people and effective degree $\bar{d} \approx 10^2$, BFS balls reach $n$ within $k \approx \log n / \log \bar{d} \approx 5$ layers — the mathematical heart of the small-world phenomenon.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| BFS/DFS algorithms, parenthesis & white-path theorems | Cormen–Leiserson–Rivest–Stein (4th ed.), Ch. 20 |
| Topological sort, SCCs (Tarjan/Kosaraju) | CLRS Ch. 20; Tarjan (1972), *SIAM J. Comput.* 1(2) |
| Connectivity, paths, cycles | Diestel (5th ed.), Ch. 1; West, Ch. 1.2 |
| Euler tours and Hierholzer's algorithm | Diestel §1.8; Bollobás, *Modern Graph Theory*, Ch. I |
| Graph metric, diameter, small worlds | Newman, *Networks* (2nd ed.), Ch. 8; Watts & Strogatz (1998), *Nature* 393 |
| Percolation and components | Bollobás & Riordan, *Percolation* (2006) |
| Receptive fields and over-squashing in GNNs | Hamilton, *Graph Representation Learning*, Ch. 5; Alon & Yahav (2021), ICLR |

**Cross-links within this repository**

- Executable BFS/DFS implementations with visualizations: [`../computation.ipynb`](../computation.ipynb)
- Weighted generalization of BFS: [`../04_shortest_paths_algorithms/`](../04_shortest_paths_algorithms/README.md)
- Spectral connectivity ($\lambda_2 \gt 0$, Fiedler theory): [`../06_graph_laplacian_and_spectral_theory/`](../06_graph_laplacian_and_spectral_theory/README.md)